# 실습 2 — AutoGluon 자동 전처리

**서울 공공자전거(따릉이) 데이터 · 모델이 읽을 수 있는 형태로 변환**

- **모델의 입력** — 숫자만
- **현실의 데이터** — `Winter` · `No Holiday` 같은 문자열 · `2017-12-01` 같은 날짜 혼재
- **전처리(Feature Engineering)** — 사람이 읽는 값 → 모델이 읽는 값으로 변환

### 이 노트북에서 확인할 것

| | 내용 | 담당 |
|---|---|---|
| 1 | 원본 확인 — 문자열·날짜 혼재 | — |
| 2 | `fit_transform` 한 줄로 자동 변환 | **자동** |
| 3 | 날짜 분해 · 범주 정수화 | **자동** |
| 4 | 타입 판별 원리 | **자동** |
| 5 | 쓸모없는 열 자동 삭제 | **자동** |
| 6 | 결측치 처리 방식 | **자동** |
| 7 | 실습 1의 발견 반영 | **사람** |
| 8 | Hour 범주형 지정 | **사람** |
| 9 | 도메인 파생 변수 | **사람** |

---

### 노트북 사용법

- 회색 배경 — 실행 가능한 **코드 셀** · 흰 배경 — 설명
- 코드 셀 선택 → **Shift + Enter** → 실행 · 결과는 하단 출력
- 상단부터 순서대로 실행

In [ ]:
# 최초 1회만 실행
# !pip install -U pip
# !pip install -U setuptools wheel
# !pip install autogluon

---
## 0. 환경 준비

- **라이브러리** — 기존에 구현된 도구 모음
- `pandas` — 표 데이터 처리 · 통상 `pd`로 표기
- `numpy` — 수치 연산 · 통상 `np`로 표기
- `AutoMLPipelineFeatureGenerator` — AutoGluon의 전처리 도구

> `import` = 불러오기 · `as` = ~라는 이름으로
> 예 — `import pandas as pd` = pandas를 불러와 pd라는 이름으로 사용

In [1]:
import pandas as pd
import numpy as np
from autogluon.features.generators import AutoMLPipelineFeatureGenerator

pd.set_option("display.max_columns", 50)

print("준비 완료")

준비 완료


---
## 1. 데이터 불러오기 · 원본 확인

- `pd.read_csv(...)` — CSV 파일 → **데이터프레임(DataFrame)** · `df`에 저장
- `encoding="latin-1"` — 컬럼명에 `°C` 등 특수문자 포함 · 미지정 시 문자 손상
- `pd.to_datetime(...)` — 문자열 날짜 → 날짜형 변환 **← 사람이 해야 하는 일**
  - 미실행 시 문자열로 남아 연·월·일·요일 분해 불가

In [2]:
CSV = "SeoulBikeData.csv"        # 경로가 다르면 여기만 수정

df = pd.read_csv(CSV, encoding="latin-1")
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")   # ← 사람

print("데이터 크기:", df.shape, "(행, 열)")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'SeoulBikeData.csv'

- `df.shape` — (행 수, 열 수) 반환
- `df.head()` — 상위 5개 행 출력 · 전체 8,760행이므로 일부만 확인

#### 확인 사항

- **문자열 값 존재** — `Seasons`의 `Winter` · `Holiday`의 `No Holiday` · `Functioning Day`의 `Yes`
  - 사람은 이해 가능 · 모델은 직접 처리 불가
- 날짜 표기 변경 — 원본 `01/12/2017` → 출력 `2017-12-01`
  - 오류 아님 · 날짜형 변환 성공의 표시

> 컬럼 14개는 한눈에 보기 어려움 → 주요 컬럼만 선별 조회
> 대괄호 `[...]` 안에 컬럼명 나열 → 해당 컬럼만 조회

In [ ]:
cols = ["Date", "Rented Bike Count", "Hour", "Temperature(°C)",
        "Seasons", "Holiday", "Functioning Day"]
df[cols].head()

---
## 2. 자동 변환 — `fit_transform`

- **예측 대상 분리** — `Rented Bike Count`는 정답이므로 전처리 대상에서 제외 **← 사람**
  - `df.drop(columns=[...])` — 지정 컬럼 제거 · 결과를 `X`에 저장
- **변환** — `AutoMLPipelineFeatureGenerator()`로 도구 생성 → `fit_transform(X)` 실행 **← 자동**

> `fit_transform` = 데이터를 학습하여(fit) 변환한다(transform)
> **전처리 전체가 이 한 줄**

In [ ]:
X = df.drop(columns=["Rented Bike Count"])   # ← 사람: 예측 대상 제외

generator = AutoMLPipelineFeatureGenerator()
X_transformed = generator.fit_transform(X)   # ← 자동: 전처리 전체

print("변환 전 컬럼 수:", X.shape[1])
print("변환 후 컬럼 수:", X_transformed.shape[1])
X_transformed.head()

> 실행 시 다수의 로그 출력 — AutoGluon의 작업 기록 · **오류 아님**
> 로그 하단에 변환된 표 출력

#### 확인 사항

- 컬럼 수 **13개 → 17개** 증가
- `Date` 분해 — `Date.year` · `Date.month` · `Date.day` · `Date.dayofweek`
- `Seasons` · `Holiday` · `Functioning Day` 문자열 → 수치 변환
- 온도·습도 등 기존 수치형 — 그대로 유지

> ⚠️ **`generator`는 재사용 불가**
> `fit_transform` 1회 실행 후 재사용 불가 → 이후 실험마다 `AutoMLPipelineFeatureGenerator()`로 **새로 생성**

### 2-1. BEFORE / AFTER 비교

- 변환 전후를 나란히 확인

In [ ]:
before_cols = ["Seasons", "Date", "Temperature(°C)"]
after_cols  = ["Seasons", "Date", "Date.year", "Date.month",
               "Date.day", "Date.dayofweek", "Temperature(°C)"]

print("===== BEFORE — 원본 (3열) =====")
display(X[before_cols].head(3))

print("===== AFTER — 전처리 후 (7열) =====")
display(X_transformed[after_cols].head(3))

> **원본 `Date`도 남음** — 삭제되지 않고 타임스탬프(1970년 기준 경과 나노초)로 변환
> 목적 — 시간의 흐름(추세)을 담기 위함
> 방침 — **일단 만들어두고 모델이 고르게 함**

---
## 3. 날짜 분해 · 범주 정수화 결과

In [ ]:
date_cols = ["Date.year", "Date.month", "Date.day", "Date.dayofweek"]
X_transformed[date_cols].head()

#### 날짜를 분해하는 이유

- 날짜 = **365개가 전부 다른 값** → 반복이 없어 규칙 학습 불가
- **요일 7개 · 월 12개**로 바꿔야 패턴이 보임
- 예 — 평일 출퇴근 수요와 주말 수요의 차이

#### 요일 번호 규칙

- pandas 기준 — 월 0 · 화 1 · 수 2 · 목 3 · **금 4** · 토 5 · 일 6
- 첫 행 값 4 = 금요일 · 2017년 12월 1일은 실제로 금요일

In [ ]:
for c in ["Seasons", "Holiday", "Functioning Day"]:
    print(f"{c:18s} {sorted(X[c].unique())}  →  {sorted(X_transformed[c].unique())}")

print()
print("2017-12-01 요일 코드:", pd.Timestamp("2017-12-01").dayofweek,
      "=", pd.Timestamp("2017-12-01").day_name())

#### 확인 사항

- 범주형 → **알파벳순** 정수 코드 부여
  - Autumn 0 · Spring 1 · Summer 2 · **Winter 3**
- 고유값 2개 → 0 / 1

> ⚠️ **번호에 크기의 의미 없음**
> Winter(3)가 Spring(1)의 3배가 아님 · 이름표를 숫자로 바꾼 것뿐

---
## 4. 타입 판별 원리

- 열 **이름**이 아니라 **값의 패턴**을 보고 판별

| 타입 | 기준 | 처리 |
|---|---|---|
| bool | 고유값이 딱 2개 | 0 / 1 |
| category | 문자열 + 값이 반복됨 | 정수 코드 (알파벳순) |
| datetime | 날짜 변환 성공 | 연·월·일·요일 분해 + 원본은 타임스탬프 |
| int / float | 이미 숫자 | 그대로 통과 |
| text | 문자열 + 대부분 고유 + 여러 단어 | 단어 빈도 · 글자수 |

In [ ]:
print(generator.feature_metadata)

#### 확인 사항

- `Seasons` → category · `Holiday` · `Functioning Day` → bool
  - 둘 다 글자인데 처리가 다름 — **고유값 개수**가 기준
- `Date` 관련 5개 → `datetime_as_int`

> ⚠️ **`Hour`는 `int`로 판별됨**
> 0~23이 숫자라서 그냥 통과 · 값의 **형태**만 볼 뿐 "시간대라는 범주"라는 **의미**는 모름
> → 8장에서 다시 확인

---
## 5. 쓸모없는 열은 자동 삭제되는가

- 상수 열 · 중복 열 · 식별자 열을 일부러 만들어 확인

| 유형 | 예시 | 왜 쓸모없나 |
|---|---|---|
| 상수 열 | 국가 — 전부 "한국" | 값이 하나뿐 → 행 구분 불가 |
| 중복 열 | 기온_사본 = 기온 | 같은 정보가 두 번 |
| 식별자 열 | 학번 — 전부 다른 값 | 외울 수는 있어도 일반화 불가 |

In [ ]:
test = X.copy()
test["국가"]     = "한국"                              # 상수 열
test["기온_사본"] = test["Temperature(°C)"]             # 중복 열
test["학번_문자"] = [f"S{i}" for i in range(len(test))]  # 식별자 (문자)
test["학번_숫자"] = range(len(test))                     # 식별자 (숫자)

out_test = AutoMLPipelineFeatureGenerator().fit_transform(test)

added    = ["국가", "기온_사본", "학번_문자", "학번_숫자"]
removed  = [c for c in added if c not in out_test.columns]
survived = [c for c in added if c in out_test.columns]

print("삭제됨   :", removed)
print("살아남음 :", survived)

#### 결과 해석

- 정보가 없는 열은 자동 삭제 — 학습 속도·메모리 절약
- 중요도 해석 왜곡 방지 — 같은 정보가 두 열로 갈리면 중요도가 절반씩 나뉨

> ⚠️ **"완전히" 같을 때만 삭제**
> 기온 ↔ 이슬점 상관 0.913 — 거의 같은 정보인데 값이 완전히 같지는 않음 → **안 지워짐**
> 이런 판단은 사람의 몫

---
## 6. 결측치(NaN) 처리 방식

- **결측치** — 값이 비어 있는 상태 · 파이썬에서 `NaN`(Not a Number)으로 표시
- 발생 원인 — 측정 누락 · 무응답 등

### 확인 방법
- `df_missing = X.copy()` — 원본 훼손 방지를 위해 복사
- `.loc[0, "Temperature(°C)"] = np.nan` — 첫 행 기온을 결측으로 변경

In [ ]:
df_missing = X.copy()
df_missing.loc[0, "Temperature(°C)"] = np.nan   # 첫 행의 기온을 비움

out_missing = AutoMLPipelineFeatureGenerator().fit_transform(df_missing)
print("변환 전:", df_missing["Temperature(°C)"].iloc[0])
print("변환 후:", out_missing["Temperature(°C)"].iloc[0], "← 채워지지 않음")

#### 결과 해석

- 결측치가 `NaN` 그대로 유지 · 평균 등으로 임의 대체되지 않음
- **일반적 방식** — 평균·중앙값으로 채움 (sklearn 대부분은 NaN 입력 시 에러)
- **AutoGluon 방식** — 채우지 않고 각 모델에 그대로 전달

| 모델 | 처리 방식 |
|---|---|
| LightGBM · XGBoost | 빈칸인 행을 어느 쪽으로 보낼지 **학습으로 결정** |
| CatBoost | 빈칸 전용 경로를 따로 둠 |
| 신경망 | 내부에서 자체 대체 |

- → 서로 다르게 처리 → **앙상블 다양성 ↑ = 성능 ↑**
- → 연구자가 사전에 대체할 필요 없음

> ⚠️ **숫자로 위장한 결측은 인식 불가** — `999` · `-1` · `0`
> 형식이 숫자라 그냥 통과 · **다음 장에서 실제 사례 처리**

---
## 7. 사람의 개입 ① — 실습 1의 발견 반영

- EDA에서 찾은 두 가지 문제를 코드로 처리

| 발견 | 층위 | 조치 |
|---|---|---|
| 습도 0%가 17건 — 위장 결측 | **값** | `0 → NaN` |
| 대여량 0이 295건 — Functioning Day와 일치 | **행** | 해당 행 제거 |

In [ ]:
# ① 위장 결측 → NaN
print("처리 전 습도 최솟값:", df["Humidity(%)"].min())
df.loc[df["Humidity(%)"] == 0, "Humidity(%)"] = np.nan
print("처리 후 습도 결측:", df["Humidity(%)"].isna().sum(), "건")
print()

# ② 누수 행 제거
d = (df[df["Functioning Day"] == "Yes"]
       .drop(columns=["Functioning Day"])
       .reset_index(drop=True))

print("행 수    %d  →  %d" % (len(df), len(d)))
print("최솟값   %d  →  %d" % (df["Rented Bike Count"].min(),
                             d["Rented Bike Count"].min()))

#### 결과 해석

- `isna()`로는 안 잡히던 결측 17건 → 정상적으로 결측 처리됨
- 대여량 최솟값 **0 → 2** — 0은 수요가 없어서가 아니라 서비스가 없어서

> **자동화가 못 하는 일** — 값의 의미 판단 · 질문에 맞는 행 선별

---
## 8. 사람의 개입 ② — `Hour` 범주형 지정

### 흔한 조언

- `Hour`(0~23)는 수치로 저장 → AutoGluon은 일반 수치로 처리
- 시간대는 크기 비교가 무의미 — "23시가 0시보다 23배"는 성립 안 함
- 즉 시간대 = 수치보다 **범주**에 가까움
- 방법 — `.astype("category")` · `astype` = 자료형 변환

In [ ]:
# 기본 처리 — Hour는 수치로 취급
out_default = AutoMLPipelineFeatureGenerator().fit_transform(X.copy())
print("기본 처리 시 Hour 타입:", out_default["Hour"].dtype)

# 사람의 개입 — 범주형으로 지정
df_hour = X.copy()
df_hour["Hour"] = df_hour["Hour"].astype("category")

out_hour = AutoMLPipelineFeatureGenerator().fit_transform(df_hour)
print("astype 적용 후 Hour 타입:", out_hour["Hour"].dtype)

#### 결과 해석

- 별도 지정 없음 → `int64` (일반 수치)
- 범주형 지정 → `category`

> ⚠️ **그런데 성능이 오르는가 — 재봐야 앎**

### 실측 결과 (AutoGluon 8개 모델 · R²)

| 모델 | Hour 숫자형 | Hour 범주형 |
|---|---|---|
| **최종 앙상블** | **0.959** | 0.955 |
| CatBoost | 0.959 | 0.952 |
| LightGBM | 0.955 | 0.951 |
| XGBoost | 0.954 | 0.938 |
| LightGBMXT | 0.950 | **0.954** |
| **신경망** | 0.942 | **0.921** |
| ExtraTrees · RandomForest | 0.932 | 0.932 |

- **8개 중 7개 하락** · 상승은 LightGBMXT 하나뿐
- 원인 — 트리는 `Hour < 6` · `Hour < 9` 로 쪼개며 **이미 봉우리를 잡고 있음**
- 열이 1개 → 24개로 늘어 각 열의 데이터가 얇아짐

> **조언이 틀린 것이 아니라 전제가 다름**
> "직선 규칙만 학습"은 **선형 모델** 이야기 · AutoGluon 주력은 트리 계열
> **전처리에 절대적 정답 없음 — 쓰는 모델을 알아야 판단 가능**

---
## 9. 사람의 개입 ③ — 도메인 파생 변수

**있는 값 → 아는 값**

| 파생 변수 | 계산 | 왜 |
|---|---|---|
| 체감온도 | 기온 + 풍속 | 5℃에 바람 불면 실제로는 0℃ |
| 불쾌지수 | 기온 + 습도 | 30℃라도 습하면 안 나감 |
| 평일 출퇴근 | 요일 + 시각 | 평일 8시와 일요일 8시는 다름 |
| 비 옴 여부 | 강수 > 0 | 1mm든 30mm든 "비 오면 안 탄다" |

> **무엇을 조합해야 의미가 있는지는 도메인 전문가만 앎**

In [ ]:
def add_fe(x):
    """도메인 파생 변수 — 사람이 만든다"""
    x = x.copy()
    T, V, H = x["Temperature(°C)"], x["Wind speed (m/s)"], x["Humidity(%)"]

    wc = 13.12 + 0.6215*T - 11.37*(V*3.6)**0.16 + 0.3965*T*(V*3.6)**0.16
    x["체감온도"]   = np.where(T <= 10, wc, T)
    x["불쾌지수"]   = 0.81*T + 0.01*H*(0.99*T - 14.3) + 46.3

    dow = x["Date"].dt.dayofweek
    x["평일출퇴근"] = ((dow < 5) & x["Hour"].isin([7,8,9,17,18,19])).astype(int)
    x["비옴"]      = (x["Rainfall(mm)"] > 0).astype(int)
    return x


d_fe = add_fe(d)
d_fe[["Temperature(°C)", "Wind speed (m/s)", "체감온도",
      "불쾌지수", "평일출퇴근", "비옴"]].head()

#### 효과 — 만들었으면 반드시 측정

| 조건 | 기본 | 파생 추가 |
|---|---|---|
| 따릉이 전체 (6,772행) | 0.954 | **0.956** |
| 500행으로 줄이면 | 0.830 | **0.852** |
| 200행으로 줄이면 | 0.746 | **0.770** |
| 선형 모델이라면 | 0.548 | **0.664** |

- 데이터가 크면 — 트리가 **알아서 조합을 찾아냄** → 거의 효과 없음
- 데이터가 작으면 — 배울 여유 없음 → **미리 만들어줘야 함**
- **만들기 전에는 알 수 없음**

> **여러분의 데이터는 대개 몇백 행**
> 심부전 299 · 산불 517 · 결근 740 · 콘크리트 1,030
> 따릉이(8,760)에서 안 통한 것이 **여러분 데이터에서는 통할 수 있음**

---
## 10. 최종 전처리 함수

- 지금까지의 처리를 하나로 묶음

In [ ]:
def preprocess(path=CSV, use_fe=True):
    x = pd.read_csv(path, encoding="latin-1")
    x["Date"] = pd.to_datetime(x["Date"], format="%d/%m/%Y")   # ① 날짜 인식

    x.loc[x["Humidity(%)"] == 0, "Humidity(%)"] = np.nan       # ② 위장 결측

    x = (x[x["Functioning Day"] == "Yes"]                      # ③ 누수 행 제거
           .drop(columns=["Functioning Day"])
           .reset_index(drop=True))

    if use_fe:
        x = add_fe(x)                                          # ④ 도메인 파생
    return x


clean = preprocess()
print("전처리 후:", clean.shape)
clean.head()

---
## 정리

### AutoGluon의 자동 처리

| 원본 데이터 | 처리 |
|---|---|
| 수치 (온도 등) | 그대로 유지 |
| 범주형 문자열 (Winter 등) | 정수로 변환 (알파벳순) |
| 참/거짓 (Yes / No) | 0 / 1로 변환 |
| 날짜 (2017-12-01) | 연·월·일·요일 분해 + 원본은 타임스탬프 |
| 문자열 (텍스트) | 단어 빈도 · 글자수 |
| 결측치 (NaN) | 대체하지 않고 모델에 전달 |
| 상수 열 · 완전 중복 열 | 자동 삭제 |

### 자동 / 사람

| AutoGluon이 알아서 | 내가 확인해야 |
|---|---|
| 글자 → 숫자 · 날짜 분해 | 숫자로 위장한 결측 (`0` · `999` · `-1`) |
| 타입 자동 판별 | 답을 이미 아는 행 (누수) |
| 상수 열 · 중복 열 삭제 | 완전히 같지 않은 중복 (상관 0.913) |
| 결측을 모델별로 처리 | 범주형 지정 여부 — **재봐야 앎** |
| 날짜 → 연·월·일·요일 파생 | 도메인 파생 변수 — 체감온도 · 출퇴근 |

---

## **형식은 자동, 의미는 사람**

> **다음 단계** — 데이터를 어떻게 나눌 것인가 (분할 구조)
> 따릉이는 시계열 · 반복 관측 데이터
> 랜덤으로 나누면 성능이 부풀려짐 → 모델 학습 파트에서 계속